In [ ]:
# ===========================================================================
# ERROR ANALYSIS - interactive driver (Phase 5)
#
# All analysis/plotting logic lives in src/error_analysis.py; this notebook
# only calls it and stores results, matching notebooks/01-04's convention.
#
# Phase 1 tells us HOW OFTEN the model is wrong. This notebook asks WHAT it
# gets wrong and WHY: which speakers and severity groups carry the errors,
# what the most confident failures look like acoustically, and whether the
# misclassified utterances differ measurably (Phase 4's Praat features) from
# the ones the model gets right. Embedding-space and attention introspection
# live in notebooks/04_model_analysis.ipynb - that's "what has the model
# learned", not "what did it get wrong".
#
# RUN SELECTION IS NOT HARDCODED ANY MORE - same repair as notebook 04. The
# previous RUN_NAME = "detection_fusion" named a run that has never existed,
# so this notebook raised on its first load_run_predictions call.
# select_analysis_run() asks the experiment registry instead.
#
# A NOTE ON WHICH RUNS ARE WORTH ANALYSING
# Error analysis is the one place a PARTIAL run is still genuinely useful:
# "which speakers does this model fail on" is answerable from whatever folds
# completed, as long as you do not read it as a performance claim. What is
# NOT useful is a run whose held-out set is single-class - there are no false
# negatives to study when no positive case was ever evaluated. The registry
# filter below excludes exactly those.
#
# PREREQUISITES
#   1. A registered run with predictions, i.e. outputs/predictions/<RUN>/*.csv.
#      Those CSVs must carry a `filename` column - predictions written before
#      src/training/reporting.py recorded utterance identity cannot be joined
#      back to audio, and load_run_predictions will say so.
#   2. outputs/praat_features.csv (notebooks/02_feature_analysis.ipynb Stage 3)
#      for the acoustic correlation in the stages below.
# ===========================================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv, print_note, print_table
from src.error_analysis import (attach_metadata, compare_error_vs_correct,
                                error_summary, fp_fn_breakdown, load_run_predictions,
                                most_confident_errors, plot_error_feature_distributions,
                                plot_error_gallery)
from src.results import select_analysis_run
from src.training.data import load_manifest
from src.training.reporting import summarize_registry

config.ensure_directories()

TASK = "detection"

# Set to a specific run to analyse it; honoured only if the registry agrees it
# is eligible. None takes the best available.
PREFERRED_RUN = None

df_m6 = load_manifest()
praat_features = (pd.read_csv(config.PRAAT_FEATURES_PATH)
                  if config.PRAAT_FEATURES_PATH.exists() else None)

registry_summary = summarize_registry()
RUN_NAME = select_analysis_run(task=TASK, metric="f1", preferred=PREFERRED_RUN)

print_header("Error Analysis")
print_kv("Manifest", f"{len(df_m6)} utterances")
print_kv("Praat features", "loaded" if praat_features is not None
         else "MISSING - run notebooks/02_feature_analysis.ipynb Stage 3 for the stages below")

if RUN_NAME is None:
    print_note(f"No eligible {TASK} run with a positive class in its held-out set. "
              "Error analysis needs both classes present - a run that only ever "
              "evaluated healthy controls has no false negatives to study. Run "
              "notebooks/03_training.ipynb further, then re-run this notebook.")
else:
    print_kv("Analysing run", RUN_NAME)
    selected = registry_summary[registry_summary["run_name"] == RUN_NAME]
    if not selected.empty:
        row = selected.iloc[0]
        print_kv("Coverage", f"{int(row['completed_folds'])}/{int(row['expected_folds'])} "
                f"folds ({row['coverage']:.0%})")
        print_kv("Status", row["status"])
        if row["status"] != "COMPLETED":
            print_note("PARTIAL run: the error PATTERNS below are informative, but the "
                      "error RATES are computed over only the folds that ran and are "
                      "not comparable against a complete run.")

In [ ]:
# STAGE 1 - Load this run's per-fold predictions and join on the manifest
# (Filepath, WordCode, Severity) plus every Phase 4 Praat feature, keyed by
# filename. `preds` is the single table every stage below reads.
preds = load_run_predictions(RUN_NAME)
preds = attach_metadata(preds, df_m6, praat_features)

print_kv("Rows", len(preds))
print_kv("Columns", len(preds.columns))
preds.head()

In [ ]:
# STAGE 2 - Where do the errors live?
#
# Overall error rate, then broken down by severity group, speaker, true class,
# and word - each sorted worst-first. The severity breakdown is the one that
# usually carries the story: if errors concentrate in 'Very Low' severity, the
# model is failing exactly where dysarthria is subtlest, which is both the
# expected result and the clinically important one to state plainly.
summary = error_summary(preds)

print_header("Overall")
print_table(summary["overall"])

print_header("Error rate by severity group")
print_table(summary["by_severity"].reset_index())

print_header("Error rate by speaker (worst 10)")
print_table(summary["by_speaker"].head(10).reset_index())

print_header("Confusions")
print_table(summary["confusions"])

for name, table in summary.items():
    table.to_csv(config.METRICS_DIR / f"errors_{RUN_NAME}_{name}.csv")
print_kv("Saved", f"{len(summary)} breakdown tables to {config.METRICS_DIR}")

In [ ]:
# STAGE 3 - The pooled confusion matrix image already written by
# notebooks/03_training.ipynb (src.training.reporting.save_confusion_matrix)
# for this exact run, displayed here alongside Stage 2's breakdown tables
# rather than redrawn.
import matplotlib.pyplot as plt

cm_path = config.CONFUSION_MATRIX_DIR / RUN_NAME / "ALL_FOLDS_pooled.png"
if cm_path.exists():
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(plt.imread(cm_path))
    ax.axis("off")
    ax.set_title(f"{RUN_NAME} — pooled confusion matrix")
    plt.show()
else:
    print_kv("Confusion matrix image", f"not found at {cm_path}")

In [ ]:
# STAGE 4 - False positive / negative analysis.
#
# fp_fn_breakdown collapses Stage 2's confusions table into an explicit
# per-class FP/FN framing: how often an actual case of this class was missed
# (false-negative rate) vs. how often a prediction of this class was wrong
# (false-positive rate) - the clinically relevant split a which-class-for-
# which-class table doesn't state on its own.
fp_fn = fp_fn_breakdown(preds)
fp_fn.to_csv(config.METRICS_DIR / f"errors_{RUN_NAME}_fp_fn.csv", index=False)

print_header("False Positive / False Negative Breakdown")
print_table(fp_fn)

In [ ]:
# STAGE 5 - Do the errors share an acoustic signature?
#
# For every Praat feature, compare the misclassified utterances against the
# correctly-classified ones (Mann-Whitney U, Bonferroni-corrected), ranked by
# Cliff's delta.
#
# Read the effect size, not the p-value: on ~21k utterances almost anything is
# 'significant', so p alone would be misleading. |delta| >= 0.33 is where a
# difference becomes worth writing about.
comparison = compare_error_vs_correct(preds)
comparison.to_csv(config.METRICS_DIR / f"errors_{RUN_NAME}_feature_comparison.csv",
                  index=False)

print_header("Errors vs correct predictions, by acoustic feature")
print_table(comparison[["feature", "mean_correct", "mean_error", "cliffs_delta",
                        "magnitude", "p_adj", "significant"]].head(12))

figure_path = plot_error_feature_distributions(preds, comparison, RUN_NAME,
                                               top_k=6, show=True)
print_kv("Figure", figure_path)
comparison

In [ ]:
# STAGE 6 - What do the worst failures actually look like?
#
# The n misclassifications the model was MOST CONFIDENT about. A wrong call at
# p=0.51 is a coin flip and tells us nothing; a wrong call at p=0.99 means the
# model has confidently learned something wrong, and that is worth looking at.
#
# Each gets a 4-panel diagnostic: waveform, spectrogram, MFCC heatmap, and
# Praat F0 contour. Figures land in outputs/figures/errors/<RUN_NAME>/.
worst = most_confident_errors(preds, n=8)
print_header("Most confident misclassifications")
print_table(worst[["filename", "speaker_id", "y_true_label",
                   "y_pred_label", "confidence"]])

gallery = plot_error_gallery(preds, RUN_NAME, n=8, show=True)

print_header("Error Analysis complete")
print_kv("Diagnostics", f"{len(gallery)} figure(s) in {config.ERROR_FIGURE_DIR / RUN_NAME}")
print_kv("Tables", f"outputs/metrics/errors_{RUN_NAME}_*.csv")